# 回测结果分析
回测表现、回撤和交易分析

In [ ]:
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from helpers import get_backtest_results

try:
    results = get_backtest_results(limit=20)
    has_data = len(results) > 0
except:
    results = pd.DataFrame()
    has_data = False

In [ ]:
# ── NAV 曲线 vs 基准 ──
if has_data:
    # 构建示例 NAV 曲线（待替换为真实 daily_nav 数据）
    dates = pd.date_range("2024-01-01", periods=120, freq="B")
    nav = (
        (1 + pd.Series(range(120), dtype=float).apply(lambda x: 0.002 * (1 + 0.3 * (x % 20 - 10) / 10).cumprod()))
        .pct_change()
        .fillna(0)
        .add(1)
        .cumprod()
    )
    benchmark = (1 + pd.Series(range(120), dtype=float) * 0.0003).pct_change().fillna(0).add(1).cumprod()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dates, y=nav, mode="lines", name="策略净值"))
    fig.add_trace(go.Scatter(x=dates, y=benchmark, mode="lines", name="基准 (沪深300)", line=dict(dash="dash")))
    fig.update_layout(
        title="策略净值 vs 基准",
        xaxis_title="日期",
        yaxis_title="累计净值",
        template="plotly_white",
    )
    fig.show()
    print(f"加载了 {len(results)} 条回测记录")
else:
    print("⚠ 暂无回测数据，无法绘制 NAV 曲线")

In [ ]:
# ── 回撤图 ──
if has_data:
    fig = go.Figure()
    # 使用示例数据绘制回撤
    drawdown = pd.Series([-0.02, -0.05, -0.03, -0.08, -0.12, -0.06, -0.03, -0.01, -0.04, -0.09])
    fig.add_trace(
        go.Scatter(
            y=drawdown * 100,
            mode="lines",
            fill="tozeroy",
            name="回撤",
            line=dict(color="rgba(220, 50, 50, 0.8)"),
            fillcolor="rgba(220, 50, 50, 0.2)",
        )
    )
    fig.update_layout(
        title="最大回撤 (示例数据)",
        xaxis_title="时间",
        yaxis_title="回撤 (%)",
        template="plotly_white",
    )
    fig.show()
else:
    print("⚠ 暂无回测数据，无法绘制回撤图")

In [ ]:
# ── 绩效指标表 ──
if has_data:
    metric_cols = [
        c
        for c in ["name", "sharpe", "max_drawdown", "annual_return", "total_return", "win_rate"]
        if c in results.columns
    ]
    if metric_cols:
        display(
            results[metric_cols]
            .head(10)
            .style.format(
                {
                    "sharpe": "{:.2f}",
                    "max_drawdown": "{:.2%}",
                    "annual_return": "{:.2%}",
                    "total_return": "{:.2%}",
                    "win_rate": "{:.2%}",
                }
            )
        )
    else:
        print("⚠ 回测结果中未找到标准指标列，显示原始数据：")
        display(results.head(10))
else:
    print("⚠ 暂无回测数据")

In [ ]:
# ── 交易散点图 ──
if has_data:
    # 示例交易数据（待替换为真实 trades 数据）
    import numpy as np

    np.random.seed(42)
    n_trades = 50
    trade_returns = np.random.normal(0.005, 0.02, n_trades)
    trade_dates = pd.date_range("2024-01-01", periods=n_trades, freq="3B")

    fig = px.scatter(
        x=trade_dates,
        y=trade_returns * 100,
        color=trade_returns,
        color_continuous_scale=["red", "white", "green"],
        labels={"x": "交易日期", "y": "收益率 (%)", "color": "收益率"},
        title="交易收益分布 (示例数据)",
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(template="plotly_white")
    fig.show()
else:
    print("⚠ 暂无回测数据，无法绘制交易散点图")

In [ ]:
# ── 空数据提示 ──
if not has_data:
    print("暂无回测数据")
    print("请确保 internal-store MCP 服务已启动 (端口 8002) 并包含回测结果")